## **Project Title**
### Analyzing Urban Mobility Patterns Using Apache Spark: A Big Data Analytics Study on Uber Trip Data

#### **Project Summary**
This project analyzes the Uber Trips Dataset using Apache Spark and PySpark. The objective is to process a large volume of ride data efficiently and identify ride demand patterns based on time factors such as hour, day, and month. By leveraging Spark's distributed computing capabilities, the project extracts meaningful business insights regarding peak travel periods, busiest days, and seasonal demand trends.

#### **Project Objectives**

The main objectives of this project are:

 + Load and process Uber trip data using PySpark.
 + Extract time-based features from ride timestamps.
 + Identify peak ride demand hours.
 + Determine the busiest days and months.
 + Visualize ride demand patterns.
 + Generate business insights from large-scale transportation data.

#### **Problem Statement**

Uber generates millions of ride requests every day. Understanding when demand is highest helps companies:

 + Allocate drivers efficiently
 + Improve customer service
 + Optimize surge pricing
 + Predict future transportation demand

This project aims to analyze Uber trip records and discover demand patterns using Big Data technologies.

#### **Tools & Technologies Used**

| Tool           | Purpose                   |
| -------------- | ------------------------- |
| Google Colab   | Development Environment   |
| Apache Spark   | Big Data Processing       |
| PySpark        | Distributed Data Analysis |
| Pandas         | Data Manipulation         |
| Plotly         | Interactive Visualization |
| Kaggle Dataset | Data Source               |


### **Business Questions**
1. Which hour has the highest ride demand?
2. Which day has the highest ride demand?
3. Which month has the highest ride demand?
4. What demand patterns can be observed?

### **Section 1: Import Libraries**

In [ ]:
!pip install pyspark plotly

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
import pandas as pd
import plotly.express as px

### **Section 2: Create Spark Session**

In [ ]:
# Initialize Spark
spark = SparkSession.builder \
    .appName("UberAnalysis") \
    .getOrCreate()

### **Section 3: Load Dataset**

In [ ]:
!wget -O uber_apr14.csv "https://raw.githubusercontent.com/fivethirtyeight/uber-tlc-foil-response/master/uber-trip-data/uber-raw-data-apr14.csv"

df = spark.read.csv(
    "/content/uber_apr14.csv",
    header=True,
    inferSchema=True
)

df.show(5)
df.printSchema()

--2026-06-16 09:07:01--  https://raw.githubusercontent.com/fivethirtyeight/uber-tlc-foil-response/master/uber-trip-data/uber-raw-data-apr14.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 26110064 (25M) [text/plain]
Saving to: ‘uber_apr14.csv’

uber_apr14.csv      100%[===================>]  24.90M  --.-KB/s    in 0.1s    

2026-06-16 09:07:02 (169 MB/s) - ‘uber_apr14.csv’ saved [26110064/26110064]

+----------------+-------+--------+------+
|       Date/Time|    Lat|     Lon|  Base|
+----------------+-------+--------+------+
|4/1/2014 0:11:00| 40.769|-73.9549|B02512|
|4/1/2014 0:17:00|40.7267|-74.0345|B02512|
|4/1/2014 0:21:00|40.7316|-73.9873|B02512|
|4/1/2014 0:28:00|40.7588|-73.9776|B02512|
|4/1/2014 0:33:00|40.7594|-73.9722|B02512|
+----------------

In [ ]:
print("Total Rows:", df.count())
print("Total Columns:", len(df.columns))
print("Column Names:", df.columns)

Total Rows: 564516
Total Columns: 4
Column Names: ['Date/Time', 'Lat', 'Lon', 'Base']


### **Section 4: Data Preprocessing**

In [ ]:
# Check Missing Values
from pyspark.sql.functions import col, sum

df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
]).show()

+---------+---+---+----+
|Date/Time|Lat|Lon|Base|
+---------+---+---+----+
|        0|  0|  0|   0|
+---------+---+---+----+



In [ ]:
# Convert Date/Time to Timestamp

from pyspark.sql.functions import to_timestamp

df = df.withColumn(
    "DateTime",
    to_timestamp(df["Date/Time"], "M/d/yyyy H:mm:ss")
)

# Converted from string to vreal time stamp: 2014-04-01 00:11:00

In [ ]:
df.printSchema()

root
 |-- Date/Time: string (nullable = true)
 |-- Lat: double (nullable = true)
 |-- Lon: double (nullable = true)
 |-- Base: string (nullable = true)
 |-- DateTime: timestamp (nullable = true)



In [ ]:
df.select("DateTime").show(5, False)

+-------------------+
|DateTime           |
+-------------------+
|2014-04-01 00:11:00|
|2014-04-01 00:17:00|
|2014-04-01 00:21:00|
|2014-04-01 00:28:00|
|2014-04-01 00:33:00|
+-------------------+
only showing top 5 rows


In [ ]:
# Extract Time Featurs (Feature extraction/Feature engineering)

from pyspark.sql.functions import hour
df = df.withColumn("Hour", hour("DateTime"))  # Extract Hour from each ride time

from pyspark.sql.functions import dayofmonth
df = df.withColumn("Day", dayofmonth("DateTime")) # Extract day of the month

from pyspark.sql.functions import month
df = df.withColumn("Month", month("DateTime")) # Extract month number

In [ ]:
df.select("DateTime", "Hour", "Day", "Month").show(5)

+-------------------+----+---+-----+
|           DateTime|Hour|Day|Month|
+-------------------+----+---+-----+
|2014-04-01 00:11:00|   0|  1|    4|
|2014-04-01 00:17:00|   0|  1|    4|
|2014-04-01 00:21:00|   0|  1|    4|
|2014-04-01 00:28:00|   0|  1|    4|
|2014-04-01 00:33:00|   0|  1|    4|
+-------------------+----+---+-----+
only showing top 5 rows


### **Section 4: EDA**
#### **Hourly Ride Demand Analysis**

In [22]:
# Which hour of the day has the highest number of Uber rides?

hourly = df.groupBy("Hour") \
           .count() \
           .orderBy("Hour")

hourly.show()

+----+-----+
|Hour|count|
+----+-----+
|   0|11910|
|   1| 7769|
|   2| 4935|
|   3| 5040|
|   4| 6095|
|   5| 9476|
|   6|18498|
|   7|24924|
|   8|22843|
|   9|17939|
|  10|17865|
|  11|18774|
|  12|19425|
|  13|22603|
|  14|27190|
|  15|35324|
|  16|42003|
|  17|45475|
|  18|43003|
|  19|38923|
+----+-----+
only showing top 20 rows


In [23]:
# Finding Peak Hour

peak_hour = df.groupBy("Hour") \
              .count() \
              .orderBy("count", ascending=False) \
              .first()

print("Peak Hour:", peak_hour["Hour"])
print("Number of Rides:", peak_hour["count"])

Peak Hour: 17
Number of Rides: 45475


The analysis shows that Hour 17 recorded the highest number of Uber ride requests. This represents the peak demand period and can help Uber optimize driver allocation and operational planning.

#### **Daily Ride Analysis**

In [24]:
# Which day of the month generates the highest trip volume?

daily = df.groupBy("Day") \
          .count() \
          .orderBy("Day")

daily.show()

+---+-----+
|Day|count|
+---+-----+
|  1|14546|
|  2|17474|
|  3|20701|
|  4|26714|
|  5|19521|
|  6|13445|
|  7|19550|
|  8|16188|
|  9|16843|
| 10|20041|
| 11|20420|
| 12|18170|
| 13|12112|
| 14|12674|
| 15|20641|
| 16|17717|
| 17|20973|
| 18|18074|
| 19|14602|
| 20|11017|
+---+-----+
only showing top 20 rows


In [25]:
# Find the Busiest Day
busiest_day = df.groupBy("Day") \
                .count() \
                .orderBy("count", ascending=False) \
                .first()

print("Busiest Day:", busiest_day["Day"])
print("Number of Rides:", busiest_day["count"])

Busiest Day: 30
Number of Rides: 36251


Day 30 recorded the highest number of Uber trips during the month. Such demand spikes may be influenced by weekday commuting patterns, public events, holidays, or local activities.

#### **Monthly Analysis**

In [26]:
# Which month generates the most trips?

monthly = df.groupBy("Month") \
            .count() \
            .orderBy("Month")

monthly.show()

+-----+------+
|Month| count|
+-----+------+
|    4|564516|
+-----+------+



Dataset containls data for only the month of April. So Monthly analysis is not possible.

#### **Convert Spark Results to Pandas**

In [27]:
pdf_hourly = hourly.toPandas()

pdf_daily = daily.toPandas()

pdf_monthly = monthly.toPandas()

In [28]:
print(pdf_hourly.head())

   Hour  count
0     0  11910
1     1   7769
2     2   4935
3     3   5040
4     4   6095


In [29]:
monthly = df.groupBy("Month").count().orderBy("Month")

pdf_hourly = hourly.toPandas()
pdf_daily = daily.toPandas()
pdf_monthly = monthly.toPandas()

print(pdf_hourly.head())

   Hour  count
0     0  11910
1     1   7769
2     2   4935
3     3   5040
4     4   6095


#### **Data Visualization**

In [30]:
# Hourly Demand

import plotly.express as px

fig = px.line(
    pdf_hourly,
    x="Hour",
    y="count",
    markers=True,
    title="Uber Demand by Hour of Day",
    labels={
        "Hour": "Hour of Day",
        "count": "Number of Trips"
    }
)

fig.show()

The hourly demand analysis indicates that Uber ride requests increase throughout the day and reach their highest level at 5 PM (Hour 17). This pattern suggests strong evening commuting activity. Ride demand is lowest during the early morning hours, particularly between 2 AM and 4 AM.

In [31]:
# Daily Demand

fig = px.bar(
    pdf_daily,
    x="Day",
    y="count",
    title="Uber Trips by Day of Month",
    labels={
        "Day": "Day of Month",
        "count": "Number of Trips"
    }
)

fig.show()

The daily demand analysis reveals that ride requests vary significantly across different days of the month. Certain days experienced noticeably higher trip volumes, indicating fluctuations in customer demand possibly driven by work schedules, events, or social activities.

In [32]:
# Montly Demand

fig = px.bar(
    pdf_monthly,
    x="Month",
    y="count",
    title="Uber Trips by Month",
    labels={
        "Month": "Month",
        "count": "Number of Trips"
    }
)

fig.show()

Since the dataset used in this project contains ride records exclusively from April 2014, monthly analysis shows activity for a single month. To perform a comparative seasonal analysis, multiple monthly datasets would need to be combined.

### **Section 5: Business Insights**

#### **Insight 1: Peak Hour**
Peak Hour: 17 (5pm)
The analysis revealed that 5 PM (Hour 17) recorded the highest number of Uber ride requests. This indicates strong evening commuter demand as people travel home from work and other daily activities.

#### **Insight 2: Hourly Demand Pattern**
Uber demand follows a clear daily pattern, increasing throughout the day and peaking during the evening commute period. Ride requests are significantly lower during late-night and early-morning hours.

#### **Insight 3: Daily Demand Variation**
Ride volumes vary across different days of the month, indicating that customer demand is not uniform. Such fluctuations may be influenced by weekdays, weekends, events, and other external factors.

#### **Conclusion**

This project successfully analyzed Uber trip data using Apache Spark and PySpark. Time-based features such as Hour, Day, and Month were extracted from ride timestamps and used to identify demand patterns. The analysis showed that the highest ride demand occurred at 5 PM, indicating strong evening commuting activity. Interactive visualizations helped present these insights effectively. The project demonstrates how Big Data technologies can be used to process large transportation datasets and support business decision-making.